In [53]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_google_genai import  ChatGoogleGenerativeAI
from dotenv import load_dotenv

import os

load_dotenv()

project =os.getenv("GOOGLE_CLOUD_PROJECT")

In [54]:

llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite",
                             vertexai=True,
                             project=project

            )

In [55]:
#prompt Template
from langchain_core.prompts import PromptTemplate

template = PromptTemplate(
    input_variables=["country"],
    template="what is the capital of {country}?"
)

In [56]:
#template.format(country="France")

In [57]:
##Langchain
chain = template | llm
response = chain.invoke({"country": "india"})
print(response)

content='The capital of India is **New Delhi**.' additional_kwargs={} response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'} id='lc_run--01a08a13-0055-7a32-b656-9bc6701c7172-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 7, 'output_tokens': 9, 'total_tokens': 16, 'input_token_details': {'cache_read': 0}}


In [58]:
print(response.content)

The capital of India is **New Delhi**.


In [59]:
template = PromptTemplate(
    input_variables=["country","topic"],
    template="what is the capital of {country} and what is the {topic}?"

)

In [60]:
##Langchain
chain = template | llm
response = chain.invoke({"country": "india","topic":"religion"})
#print(response.content)
#print(response)
response.pretty_print()


================================== Ai Message ==================================

The capital of India is **New Delhi**.

India does not have a single state religion. It is a **secular country**, meaning it does not officially endorse any particular religion. However, the largest religion practiced in India is **Hinduism**, followed by Islam, Sikhism, Buddhism, Christianity, and Jainism, among others.


few shot prompting

In [61]:
#few shot example  in a raw way and using langchain
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
Messages = [
    SystemMessage(content="You are a helpful assistant that answers geography questions."),
    HumanMessage(content="What is the capital of France?"),
    AIMessage(content="The capital of France is Paris."),
    HumanMessage(content="What is the capital of India?")
]

In [62]:
response = llm.invoke(Messages)

In [63]:
print(response.content)

The capital of India is **New Delhi**.


# chain = llm | Messages ❌ → `|` connects Runnable components, but Messages is only input data (a Python list).
# Correct flow: Messages (input data) → llm.invoke(Messages) → LLM processes messages → Response ✅



# `|` connects one Runnable component to another Runnable component.
# Messages is only input data (a Python list), not a Runnable.

# ❌ Wrong:  LLM → Messages
chain = llm | Messages

# ✅ Correct: Messages (input) → LLM → Response
response = llm.invoke(Messages)

# ✅ Chain example: Input → Prompt (Runnable) → LLM (Runnable) → Response
chain = prompt | llm

In [64]:
# in the above case #chain = llm | Messages will not work because the Messages is a list of messages and not a prompt template.

In [76]:
#Another few shot example  in a raw way and using langchain
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
Messages = [
    SystemMessage(content="you are a  math teacher for primary school"),
    HumanMessage(content="What is 2 + 2?"),
    AIMessage(content ="This refers to addition \n count numbers on fingers \n 2 + 2 = 4"),
    HumanMessage(content = "2 *2 = ?"),
    AIMessage(content = "This refers to multiplication \n count numbers on fingers \n 2 * 2 = 4"),
    HumanMessage(content = "10-4 = ?")
]

In [77]:
response = llm.invoke(Messages)

In [78]:
print (response.content)



This refers to subtraction 
 count numbers on fingers 
 10 - 4 = 6


In [70]:
#multiple prompts in dynamic way using prompt templates

from langchain_core.prompts import ChatPromptTemplate
prompt = ChatPromptTemplate.from_messages([
    ("system", "you are a math teacher for primary school"),
    ("human", "2*2 =?"),
    ("ai", "This refers to multiplication \n Ensure you have memorized the tables \n" ),
    ("human", "5+2 =?"),
    ("ai", "This refers to addition \n count numbers on fingers \n 2 + 2 = 4" ),
    ("human", "{question}")
]

)

In [82]:
chain = prompt| llm
response = chain.invoke({"question": "10*5"})

In [84]:
print(response.content)

This refers to multiplication.
You can think of this as adding 10 five times:
10 + 10 + 10 + 10 + 10 = 50

Or, you can think of it as adding 5 ten times:
5 + 5 + 5 + 5 + 5 + 5 + 5 + 5 + 5 + 5 = 50

So, 10 * 5 = **50**


In [83]:
response.pretty_print()

================================== Ai Message ==================================

This refers to multiplication.
You can think of this as adding 10 five times:
10 + 10 + 10 + 10 + 10 = 50

Or, you can think of it as adding 5 ten times:
5 + 5 + 5 + 5 + 5 + 5 + 5 + 5 + 5 + 5 = 50

So, 10 * 5 = **50**
